## Setup
Install required libraries for fine-tuning our Legal BERT model on the CUAD dataset.

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

## Load and Prepare Dataset
Fetches the CUAD dataset from HuggingFace and maps clause types to our three Risk Levels (Low, Medium, High).

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset

print("Loading dataset...")
dataset = load_dataset("theatticusproject/cuad")

# Comprehensive mapping based on the project data loader
risk_mapping = {
    'Limitation of Liability': 'High', 'Indemnification': 'High', 'IP Ownership Assignment': 'High', 'Change of Control': 'High', 'Exclusivity': 'High', 'Non-Compete': 'High', 'No-Solicit of Customers': 'High', 'No-Solicit of Employees': 'High', 'Most Favored Nation (MFN)': 'High', 'Liquidated Damages': 'High', 'Revenue/Profit Sharing': 'High', 'Cap on Liability': 'High', 'Uncapped Liability': 'High', 'Price Restriction': 'High', 'Volume Restriction': 'High',
    'Termination for Convenience': 'Medium', 'Audit Rights': 'Medium', 'Confidentiality': 'Medium', 'License Grant': 'Medium', 'Non-Disparagement': 'Medium', 'Anti-Assignment': 'Medium', 'Insurance': 'Medium', 'Minimum Commitment': 'Medium', 'Security Breach Statement': 'Medium', 'Warranties': 'Medium', 'Irrevocable or Perpetual License': 'Medium', 'Source Code Escrow': 'Medium',
    'Governing Law': 'Low', 'Effective Date': 'Low', 'Expiration Date': 'Low', 'Renewal Term': 'Low', 'Notice Period to Terminate Renewal': 'Low', 'Document Name': 'Low', 'Parties': 'Low', 'Agreement Date': 'Low', 'Covenant not to Sue': 'Low', 'Third Party Beneficiary': 'Low', 'Force Majeure': 'Low', 'Affiliate License-Licensor': 'Low', 'Affiliate IP License-Licensee': 'Low', 'Joint IP Ownership': 'Low', 'Non-Transferable License': 'Low'
}

def map_to_risk(qas):
    records = []
    for qa in qas:
        question_text = qa['question']
        risk_level = 'Low' # Default
        
        # Check if any high/medium risk keywords are IN the question
        for key, level in risk_mapping.items():
            if key.lower() in question_text.lower():
                risk_level = level
                break # Found the mapping
                
        for answer in qa['answers']:
            if answer['text'].strip():
                records.append({'text': answer['text'], 'label': risk_level})
    return records

processed_train = []
for item in dataset['train']:
    processed_train.extend(map_to_risk(item['qas']))

processed_test = []
for item in dataset['test']:
    processed_test.extend(map_to_risk(item['qas']))

label_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_train = pd.DataFrame(processed_train)
df_test = pd.DataFrame(processed_test)

print("\n--- Verifying Label Distribution ---")
print("Training Set Distribution:")
print(df_train['label'].value_counts())
print("\nTest Set Distribution:")
print(df_test['label'].value_counts())
print("------------------------------------\n")

df_train['label_idx'] = df_train['label'].map(label_map)
df_test['label_idx'] = df_test['label'].map(label_map)

train_ds = Dataset.from_pandas(df_train)
test_ds = Dataset.from_pandas(df_test)
print(f"\nTrain records: {len(train_ds)}, Test records: {len(test_ds)}")


## Tokenization

In [ ]:
from transformers import AutoTokenizer

model_id = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_fn(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=512)

print("Tokenizing dataset...")
tokenized_train = train_ds.map(tokenize_fn, batched=True)
tokenized_test = test_ds.map(tokenize_fn, batched=True)

tokenized_train = tokenized_train.remove_columns(['text', 'label'])
tokenized_train = tokenized_train.rename_column('label_idx', 'labels')
tokenized_train.set_format('torch')

tokenized_test = tokenized_test.remove_columns(['text', 'label'])
tokenized_test = tokenized_test.rename_column('label_idx', 'labels')
tokenized_test.set_format('torch')
print("Done!")


## Fine-tune the Model

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

print("Starting training...")
trainer.train()


## Export and Zip the Model

In [ ]:
save_dir = "legal_risk_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Model saved globally! Zipping now...")

!zip -r legal_risk_model.zip legal_risk_model/
print("\nModel zipped! Please download 'legal_risk_model.zip' from the Colab file explorer, extract it, and place the folder in your local project under 'models/legal_risk_model'.")
